<a href="https://colab.research.google.com/github/Gowtham13042007/cron_job/blob/main/Fake_newsipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import numpy as np
import pandas as pd
import re
import nltk #natural language toolkit
from nltk.corpus import stopwords  # the is and are
from nltk.stem.porter import PorterStemmer # converts words to root(playing to play)
from sklearn.feature_extraction.text import TfidfVectorizer #text to numerical vectors
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("Downloading NLTK stopwords")
nltk.download('stopwords')

df_fake = pd.read_csv('/content/Fake.csv', engine='python', on_bad_lines='skip')
df_real = pd.read_csv('/content/True.csv',engine='python', on_bad_lines='skip')

df_fake['label'] = 1
df_real['label'] = 0

df_combined = pd.concat([df_fake, df_real], ignore_index=True)
X = df_combined[['title','text','subject']]  # Change 'text' to whatever your text column is named
y = df_combined['label']

print(X.isnull().sum())

print('Stemming Text Data')
port_stem=PorterStemmer()

def stemming(content):
  stemmed_content=re.sub('[^a-zA-Z]',' ',content) #remove everything that is not lowercase or upercase letter
  stemmed_content=stemmed_content.lower() # convert all text to lowercase
  stemmed_content=stemmed_content.split()  # text into single words
  stemmed_content=[port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content=' '.join(stemmed_content)
  return stemmed_content

X['text']=X['text'].apply(stemming)

x=X['text'].values
y=df_combined['label'].values

print('Transforming text into Numerical Vectors')
vectorizer=TfidfVectorizer()
vectorizer.fit(x)

x = vectorizer.transform(x)
X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=2)

model = LogisticRegression()
model.fit(X_train, Y_train)
print("Model training completed.")

print("\n--- Model Evaluation ---")
# Metrics on Training Data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)
print(f"Accuracy score of the training data : {training_data_accuracy * 100:.2f}%")

# Metrics on Testing Data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)
print(f"Accuracy score of the test data     : {test_data_accuracy * 100:.2f}%")     #MultinomialNB treats every word as completely unrelated to the next. Fake news often relies on structural context, subtle sarcasm, or phrases like "not a true statement".


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


title      0
text       0
subject    0
dtype: int64
Stemming Text Data


/tmp/ipykernel_2782/484538834.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['text']=X['text'].apply(stemming)


Transforming text into Numerical Vectors
Model training completed.

--- Model Evaluation ---
Accuracy score of the training data : 99.54%
Accuracy score of the test data     : 98.52%
